# SM-SIP: Few-Shot Inference & Evaluation

## Semantic & Multilingual Salient Information Prompting

This notebook implements the **few-shot inference pipeline** for Italian text summarization:

1. **Load** the fine-tuned SigExt model from Hugging Face
2. **Generate** a held-out test set (never seen during training)
3. **Run** the full pipeline with **in-context examples**: SigExt → Llama-3 → Evaluation
4. **Export** results to JSON for analysis

### Key Difference from Zero-Shot
The few-shot approach provides **example summaries** in the prompt, helping the model understand:
- The expected output format
- The desired summary length
- How to incorporate keyphrases naturally

### Configuration
- **SigExt Model**: `LookUpMark/sigext-wits-it-10k`
- **LLM**: `meta-llama/Meta-Llama-3-8B-Instruct` (4-bit quantized)
- **Test Samples**: 50 articles (skipping training data)

---
## 1. Environment Setup

Install all required dependencies and authenticate with Hugging Face.

In [1]:
# Install dependencies
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \
    huggingface_hub 'numpy<2.0' 'scipy>=1.10'

# Download Italian Spacy model
!python -m spacy download it_core_news_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/it_core_news_sm-3.8.0/it_core_news_sm-3.8.0-py3-none-any.whl (13.0 MB)
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')


In [2]:
import os
import torch
import json
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

/home/marcantoniolopez/Documenti/github/projects/DNLPProj/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## 2. Configuration

Define model paths and authentication.

Token is loaded from:
1. **Kaggle**: Using `kaggle_secrets` (for Kaggle notebooks)
2. **Environment**: Using `HF_TOKEN` environment variable (for local/Colab)

In [3]:
# Hugging Face authentication
# Option 1: Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    # Option 2: Environment variable or manual input
    HF_TOKEN = os.getenv("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"

# Configuration
CONFIG = {
    "SIGEXT_MODEL_ID": "LookUpMark/sigext-wits-it-25k",
    "LLAMA_MODEL_ID": "meta-llama/Llama-3.1-8B-Instruct",
    "NUM_TEST_SAMPLES": 50,
    "MAX_LEN": 2048,
    "SKIP_TRAIN_SAMPLES": 25000  # Skip first 25k samples used for training
}

# Authenticate with Hugging Face
login(token=HF_TOKEN)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


---
## 3. Generate Held-Out Test Set

Load the WITS dataset and create a test set that was **never seen during training**.
We skip the first 10,000 samples to avoid data contamination.

In [4]:
def get_test_data():
    """Generate a held-out test set from WITS dataset."""
    print(f"--- Generating Test Set ({CONFIG['NUM_TEST_SAMPLES']} new articles) ---")
    print(f"    Skipping first {CONFIG['SKIP_TRAIN_SAMPLES']} samples to avoid contamination.")
    
    # Load dataset in streaming mode
    dataset = load_dataset("silvia-casola/WITS", split="train", streaming=True)
    
    # Skip training data
    dataset = dataset.skip(CONFIG['SKIP_TRAIN_SAMPLES'])
    
    test_data = []
    print("    Downloading and filtering...")
    pbar = tqdm(total=CONFIG['NUM_TEST_SAMPLES'])
    
    for entry in dataset:
        source = entry['source']
        summary = entry['summary']
        
        # Quality filters
        if len(source) < 500 or len(summary) < 50 or len(source) > 10000:
            continue
            
        test_data.append({
            "source": source,
            "reference": summary
        })
        pbar.update(1)
        
        if len(test_data) >= CONFIG['NUM_TEST_SAMPLES']:
            break
            
    pbar.close()
    print(f"    Test Set ready: {len(test_data)} examples.")
    return test_data

test_data = get_test_data()

--- Generating Test Set (50 new articles) ---
    Skipping first 25000 samples to avoid contamination.


Repo card metadata block was not found. Setting CardData to empty.


100%|██████████| 50/50 [00:18<00:00,  2.70it/s]

    Test Set ready: 50 examples.


---
## 4. Load Models

### 4.1 SigExt Model
Load the fine-tuned Salient Information Extractor from Hugging Face.

### 4.2 Llama-3 (4-bit Quantized)
Load Llama-3-8B-Instruct with 4-bit quantization for efficient inference on T4 GPU.

In [5]:
def load_models():
    """Load SigExt and Llama-3 models."""
    print("--- Loading Models ---")
    
    # A. Load SigExt from Hugging Face
    print(f"1. Downloading SigExt: {CONFIG['SIGEXT_MODEL_ID']}...")
    try:
        sigext_tokenizer = AutoTokenizer.from_pretrained(CONFIG['SIGEXT_MODEL_ID'])
        sigext_model = AutoModelForTokenClassification.from_pretrained(
            CONFIG['SIGEXT_MODEL_ID']
        ).to("cpu")
        print("    -> SigExt loaded successfully!")
    except Exception as e:
        print(f"    ERROR loading SigExt: {e}")
        print("    Verify the repo exists and is accessible with your token.")
        raise e

    # B. Load Llama-3 (4-bit Quantized)
    print(f"2. Loading Llama-3: {CONFIG['LLAMA_MODEL_ID']} (4-bit)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, 
        bnb_4bit_quant_type="nf4", 
        bnb_4bit_compute_dtype=torch.float16, 
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=True
    )
    llama_model = AutoModelForCausalLM.from_pretrained(
        CONFIG['LLAMA_MODEL_ID'], 
        quantization_config=bnb_config, 
        device_map="auto"
    )
    llama_tokenizer = AutoTokenizer.from_pretrained(CONFIG['LLAMA_MODEL_ID'])
    llama_tokenizer.pad_token = llama_tokenizer.eos_token
    print("    -> Llama-3 loaded successfully!")
    
    return sigext_model, sigext_tokenizer, llama_model, llama_tokenizer

sigext_model, sigext_tokenizer, llama_model, llama_tokenizer = load_models()

--- Loading Models ---
1. Downloading SigExt: LookUpMark/sigext-wits-it-25k...
    -> SigExt loaded successfully!
2. Loading Llama-3: meta-llama/Llama-3.1-8B-Instruct (4-bit)...


Loading checkpoint shards: 100%|██████████| 4/4 [00:31<00:00,  7.92s/it]


    -> Llama-3 loaded successfully!


---
## 5. LangChain Pipeline Setup (Few-Shot)

Configure the LangChain pipeline with a **few-shot prompt template**.

### Why Few-Shot?
- Provides concrete examples of desired output format
- Demonstrates how to integrate keyphrases naturally
- Improves summary quality and consistency
- Reduces hallucination by showing expected length/style

In [6]:
def setup_few_shot_chain(llama_model, llama_tokenizer):
    """Create a few-shot LangChain pipeline with in-context examples."""
    
    # Create HuggingFace pipeline
    pipe = pipeline(
        "text-generation", 
        model=llama_model, 
        tokenizer=llama_tokenizer, 
        max_new_tokens=256, 
        temperature=0.1
    )
    
    # Wrap in LangChain
    llm = HuggingFacePipeline(pipeline=pipe)
    
    # Few-shot prompt template with in-context examples
    template = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert academic summarizer for Italian texts. You must include the provided keyphrases naturally in your summary.
<|eot_id|><|start_header_id|>user<|end_header_id|>

### Example 1:
Text: "Roma, fondata secondo la tradizione il 21 aprile 753 a.C., è la capitale d'Italia. Con i suoi quasi tre milioni di abitanti è il comune più popoloso d'Italia. Il centro storico, patrimonio UNESCO, ospita monumenti come il Colosseo, il Pantheon e la Fontana di Trevi."

Keyphrases: Roma capitale Italia, Colosseo, patrimonio UNESCO

Summary: "Roma è la capitale d'Italia e il comune più popoloso del paese. Il suo centro storico, riconosciuto patrimonio UNESCO, include monumenti iconici come il Colosseo."

### Example 2:
Text: "L'intelligenza artificiale sta trasformando il settore sanitario. Gli algoritmi di machine learning possono analizzare immagini mediche per rilevare tumori con precisione superiore ai radiologi umani. Questa tecnologia promette diagnosi più rapide e accurate."

Keyphrases: intelligenza artificiale, settore sanitario, diagnosi, machine learning

Summary: "L'intelligenza artificiale sta rivoluzionando il settore sanitario. Attraverso il machine learning, gli algoritmi analizzano immagini mediche per diagnosi più precise e veloci."

---

### Your Task:
Original Text:
{source}

CONTROL INSTRUCTIONS:
It is essential to include the following key concepts in the summary:
{keyphrases}

Generate a summary in Italian (similar length and style to the examples above):<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
    
    prompt = PromptTemplate(template=template, input_variables=["source", "keyphrases"])

    chain = prompt | llm | StrOutputParser()
    
    print("Few-shot chain configured successfully!")
    return chain

chain = setup_few_shot_chain(llama_model, llama_tokenizer)

Device set to use cuda:0


Few-shot chain configured successfully!


---
## 6. Keyphrase Extraction Function

Use the SigExt model to extract salient tokens from the source text.
These keyphrases will be injected into the LLM prompt to guide generation.

In [7]:
def extract_keyphrases(text, model, tokenizer):
    """Extract salient keyphrases using SigExt model."""
    # Tokenize input
    inputs = tokenizer(
        text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=CONFIG["MAX_LEN"]
    ).to("cpu")
    
    # Run inference
    with torch.no_grad():
        logits = model(**inputs).logits
    
    # Get predictions (0 = non-salient, 1 = salient)
    preds = torch.argmax(logits, dim=2)[0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    # Extract tokens marked as salient
    extracted_tokens = [t for t, label in zip(tokens, preds) if label == 1]
    
    # Decode to text
    decoded = tokenizer.decode(tokenizer.convert_tokens_to_ids(extracted_tokens))
    return decoded

---
## 7. Evaluation Metrics

Evaluate the generated summaries using three complementary metrics:

- **ROUGE-1**: Lexical overlap (unigram precision/recall)
- **BERTScore**: Semantic similarity using BERT embeddings
- **KIR (Keyphrase Inclusion Rate)**: Measures prompt obedience

In [8]:
def evaluate(test_data, sigext_model, sigext_tokenizer, chain):
    """Run full evaluation on test set."""
    print(f"--- Starting Evaluation on {len(test_data)} articles ---")
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    
    metrics = {"bert": [], "rouge": [], "kir": []}
    
    print("    Processing (Llama-3 is generating)...")
    for item in tqdm(test_data):
        try:
            # 1. SigExt: Extract keyphrases
            keys_text = extract_keyphrases(item['source'], sigext_model, sigext_tokenizer)
            keys_list = [k.strip() for k in keys_text.split() if len(k) > 3]
            
            # 2. Llama-3: Generate summary
            res = chain.invoke({"source": item['source'], "keyphrases": keys_text})
            gen_summary = res.split("assistant<|end_header_id|>")[-1].strip()
            
            # 3. Calculate metrics
            # ROUGE-1
            metrics["rouge"].append(
                scorer.score(item['reference'], gen_summary)['rouge1'].fmeasure
            )
            
            # BERTScore
            _, _, F1 = bert_score(
                [gen_summary], [item['reference']], lang="it", verbose=False
            )
            metrics["bert"].append(F1.mean().item())
            
            # KIR (Keyphrase Inclusion Rate)
            if keys_list:
                gen_lower = gen_summary.lower()
                hits = sum(1 for k in keys_list if k.lower() in gen_lower)
                metrics["kir"].append(hits / len(keys_list))
            else:
                metrics["kir"].append(0.0)
                
        except Exception as e:
            print(f"    ! Error on sample: {e}")
            continue

    return metrics

---
## 8. Run Evaluation & Export Results

Run the evaluation and save results to a JSON file for later analysis.

In [9]:
# Run evaluation
metrics = evaluate(test_data, sigext_model, sigext_tokenizer, chain)

# Compute statistics and build results object
results = {
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "inference_type": "few-shot",
        "sigext_model": CONFIG["SIGEXT_MODEL_ID"],
        "llm_model": CONFIG["LLAMA_MODEL_ID"],
        "num_test_samples": CONFIG["NUM_TEST_SAMPLES"],
        "skip_train_samples": CONFIG["SKIP_TRAIN_SAMPLES"]
    },
    "metrics": {
        "bert_score": {
            "mean": float(np.mean(metrics['bert'])),
            "std": float(np.std(metrics['bert'])),
            "min": float(np.min(metrics['bert'])),
            "max": float(np.max(metrics['bert']))
        },
        "rouge1": {
            "mean": float(np.mean(metrics['rouge'])),
            "std": float(np.std(metrics['rouge'])),
            "min": float(np.min(metrics['rouge'])),
            "max": float(np.max(metrics['rouge']))
        },
        "kir": {
            "mean": float(np.mean(metrics['kir'])),
            "std": float(np.std(metrics['kir'])),
            "min": float(np.min(metrics['kir'])),
            "max": float(np.max(metrics['kir']))
        }
    },
    "raw_scores": {
        "bert": [float(x) for x in metrics['bert']],
        "rouge": [float(x) for x in metrics['rouge']],
        "kir": [float(x) for x in metrics['kir']]
    }
}

# Save to JSON
output_file = f"results_few_shot_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

# Display results
print("\n" + "="*55)
print("   FEW-SHOT INFERENCE RESULTS")
print("="*55)
print(f"   BERTScore: {results['metrics']['bert_score']['mean']:.4f} ± {results['metrics']['bert_score']['std']:.4f}")
print(f"   ROUGE-1:   {results['metrics']['rouge1']['mean']:.4f} ± {results['metrics']['rouge1']['std']:.4f}")
print(f"   KIR:       {results['metrics']['kir']['mean']:.2%} ± {results['metrics']['kir']['std']:.2%}")
print("="*55)
print(f"\n   Results saved to: {output_file}")

--- Starting Evaluation on 50 articles ---
    Processing (Llama-3 is generating)...


 76%|███████▌  | 38/50 [08:23<02:22, 11.90s/it]

    ! Error on sample: CUDA out of memory. Tried to allocate 150.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 62.75 MiB is free. Process 4315 has 240.76 MiB memory in use. Process 8936 has 44.40 MiB memory in use. Including non-PyTorch memory, this process has 6.87 GiB memory in use. Of the allocated memory 6.39 GiB is allocated by PyTorch, and 370.66 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


 82%|████████▏ | 41/50 [08:56<01:37, 10.83s/it]

    ! Error on sample: CUDA out of memory. Tried to allocate 146.00 MiB. GPU 0 has a total capacity of 7.62 GiB of which 77.38 MiB is free. Process 4315 has 241.57 MiB memory in use. Process 8936 has 44.40 MiB memory in use. Including non-PyTorch memory, this process has 6.87 GiB memory in use. Of the allocated memory 6.40 GiB is allocated by PyTorch, and 356.64 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)


100%|██████████| 50/50 [10:59<00:00, 13.20s/it]


   FEW-SHOT INFERENCE RESULTS
   BERTScore: 0.6600 ± 0.0453
   ROUGE-1:   0.1903 ± 0.0921
   KIR:       29.63% ± 14.68%

   Results saved to: results_few_shot_20251223_160107.json


---
## 9. Comparison: Zero-Shot vs Few-Shot

Run both notebooks and compare the results:

| Metric | Zero-Shot | Few-Shot |
|--------|-----------|----------|
| BERTScore | ___ | ___ |
| ROUGE-1 | ___ | ___ |
| KIR | ___ | ___ |

### Expected Observations
- **Few-shot should show higher KIR**: Examples teach the model to incorporate keyphrases
- **More consistent summary length**: Examples provide a length reference
- **Better formatting**: Examples demonstrate desired structure